# Benchmark: Parallel Scaling

Loads pre-computed data from `benchmark_parallel_cores.csv` (generated by
`run_benchmark_parallel.py`) and produces a single plot:

**Cores sweep** — pairs/s vs number of cores, ±1 std shaded band, with a dashed
ideal-linear-scaling reference.

The naive vs. standard vs. precomputed-keys comparison lives in
`benchmark_precompute.ipynb`.

## Reproducing the data

The CSV consumed by this notebook lives under `data/paper/`, which is **not checked into git**. Regenerate it with:

```bash
pixi run -e dev python scripts/run_benchmark_parallel.py
```


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CORES_FILE = "../../data/paper/benchmark_parallel_cores.csv"

In [ ]:
df_cores = pd.read_csv(CORES_FILE)

xs      = df_cores["n_cores"].to_numpy()
mean_pps = df_cores["mean_pps"].to_numpy()
std_pps  = df_cores["std_pps"].to_numpy()

In [ ]:
from pathlib import Path

color = "#2196F3"  # blue

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(xs, mean_pps, color=color, marker="o", markersize=5, linewidth=1.5, label="pairs/s")
ax.fill_between(xs, mean_pps - std_pps, mean_pps + std_pps, color=color, alpha=0.2)

ideal = mean_pps[0] * xs
ax.plot(xs, ideal, color="grey", linewidth=1, linestyle="--", label="ideal linear scaling")

ax.set_xlabel("Number of cores", fontsize=15)
ax.set_ylabel("Pairs / second", fontsize=15)
ax.set_yscale("log")
ax.set_xscale("log", base=2)
ax.set_xticks(xs)
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.tick_params(axis="both", labelsize=13)
ax.legend(fontsize=13)
ax.set_xlim(xs[0] * 0.8, xs[-1] * 1.2)
ax.set_ylim(bottom=0)
ax.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()

out = Path("../../smartreact_paper/figures/parallelisation.png")
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out, dpi=300, bbox_inches="tight")
print(f"Saved {out}")
plt.show()

## Summary table

In [ ]:
df_cores.assign(
    speedup=(df_cores["mean_pps"] / df_cores["mean_pps"].iloc[0]).round(2),
    mean_pps=df_cores["mean_pps"].round(1),
    std_pps=df_cores["std_pps"].round(1),
)